In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

In [ ]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

DB_PATH = "results/variants.db"

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

print("Connected to:", DB_PATH)
from IPython.display import display, clear_output
import plotly.io as pio
pio.renderers.default = "notebook"


## Cross-sample QC overview
Mapping rate, variant counts, and mean depth for every run in the database.

In [ ]:
runs = query("""
    SELECT r.run_id, s.name AS sample, s.cell_line,
           r.mapping_rate, r.total_raw, r.total_filt, r.run_date
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY s.cell_line, r.run_id
""")

if runs.empty:
    print("No runs found in database.")
else:
    runs["label"] = runs["cell_line"].fillna(runs["sample"])
    runs_sorted = runs.sort_values("mapping_rate", ascending=True)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Mapping rate (%) — all runs", "Filtered variants per run"),
        horizontal_spacing=0.12
    )

    # Horizontal bar — mapping rate, coloured by pass/fail
    colors = ["#e63946" if v < 85 else "#2d6a4f" for v in runs_sorted["mapping_rate"]]
    fig.add_trace(go.Bar(
        y=runs_sorted["label"],
        x=runs_sorted["mapping_rate"],
        orientation="h",
        marker_color=colors,
        hovertemplate="%{y}<br>Mapping rate: %{x:.1f}%<extra></extra>"
    ), row=1, col=1)
    fig.add_vline(x=85, line_dash="dash", line_color="#e63946",
                  annotation_text="85%", row=1, col=1)

    # Horizontal bar — filtered variants
    runs_sorted2 = runs.sort_values("total_filt", ascending=True)
    fig.add_trace(go.Bar(
        y=runs_sorted2["label"],
        x=runs_sorted2["total_filt"],
        orientation="h",
        marker_color="#457b9d",
        hovertemplate="%{y}<br>Filtered variants: %{x:,}<extra></extra>"
    ), row=1, col=2)

    n = len(runs)
    fig.update_layout(
        height=max(400, 18 * n),
        title=f"Run QC overview — {n} runs",
        showlegend=False
    )
    fig.update_xaxes(range=[0, 105], row=1, col=1)
    fig.show()

    # Summary table grouped by cell line
    summary = (
        runs.groupby("label").agg(
            n_runs=("run_id", "count"),
            mean_mapping=("mapping_rate", "mean"),
            mean_filt_vars=("total_filt", "mean"),
        ).reset_index().rename(columns={"label": "cell_line"})
        .sort_values("mean_filt_vars", ascending=False)
    )
    display(summary.style.format({
        "mean_mapping": "{:.1f}",
        "mean_filt_vars": "{:,.0f}"
    }))


## Per-run deep-dive
Select a run to explore depth distribution, allele frequencies, QUAL scores, and chromosomal variant distribution.

In [ ]:
run_ids = runs['run_id'].tolist() if not runs.empty else []

run_selector = widgets.Dropdown(
    options=run_ids,
    description='Run:',
    layout=widgets.Layout(width='350px')
)

_perrun_busy = [False]

def render_perrun(run_id):
    if _perrun_busy[0]:
        return
    _perrun_busy[0] = True
    try:
        clear_output(wait=True)
        display(run_selector)

        calls = query("""
            SELECT gc.depth, gc.quality, gc.allele_freq,
                   v.variant_type, v.chromosome
            FROM genotype_calls gc
            JOIN variants v ON gc.variant_id = v.variant_id
            WHERE gc.run_id = ?
        """, (run_id,))

        if calls.empty:
            print(f'No variant calls found for {run_id}.')
            return

        main_chroms = [f'chr{i}' for i in list(range(1, 23)) + ['X', 'Y', 'M']]
        calls_main = calls[calls['chromosome'].isin(main_chroms)]
        type_counts = calls['variant_type'].value_counts().reset_index()
        type_counts.columns = ['type', 'count']

        fig = make_subplots(
            rows=2, cols=3,
            subplot_titles=(
                'Read depth distribution', 'QUAL score distribution', 'Variant types',
                'Allele frequency spectrum', 'Variants per chromosome', 'Depth vs QUAL'
            ),
            specs=[
                [{'type': 'xy'}, {'type': 'xy'}, {'type': 'domain'}],
                [{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}]
            ],
            horizontal_spacing=0.10, vertical_spacing=0.15
        )

        depth_cap = calls['depth'].quantile(0.99)
        fig.add_trace(go.Histogram(
            x=calls[calls['depth'] <= depth_cap]['depth'],
            nbinsx=50, marker_color='#457b9d', name='Depth'
        ), row=1, col=1)
        fig.add_trace(go.Histogram(
            x=calls['quality'], nbinsx=50,
            marker_color='#2d6a4f', name='QUAL'
        ), row=1, col=2)
        fig.add_trace(go.Pie(
            labels=type_counts['type'],
            values=type_counts['count'],
            hole=0.4,
            marker_colors=['#2d6a4f', '#e63946', '#457b9d', '#f4a261']
        ), row=1, col=3)
        fig.add_trace(go.Histogram(
            x=calls['allele_freq'], nbinsx=40,
            marker_color='#f4a261', name='AF'
        ), row=2, col=1)

        chrom_counts = (
            calls_main.groupby('chromosome').size()
            .reindex(main_chroms).fillna(0).reset_index()
        )
        chrom_counts.columns = ['chromosome', 'count']
        fig.add_trace(go.Bar(
            x=chrom_counts['chromosome'], y=chrom_counts['count'],
            marker_color='#6d6875', name='Variants'
        ), row=2, col=2)

        sample_n = min(5000, len(calls))
        scatter_df = calls.sample(sample_n, random_state=42)
        fig.add_trace(go.Scatter(
            x=scatter_df['depth'], y=scatter_df['quality'],
            mode='markers',
            marker=dict(size=3, opacity=0.4, color='#1d3557'),
            name='calls'
        ), row=2, col=3)

        fig.update_layout(
            height=700,
            title=f'Per-run deep-dive: {run_id}  ({len(calls):,} variant calls)',
            showlegend=False
        )
        fig.update_xaxes(title_text='Depth', row=1, col=1)
        fig.update_xaxes(title_text='QUAL', row=1, col=2)
        fig.update_xaxes(title_text='Allele frequency', row=2, col=1)
        fig.update_xaxes(tickangle=45, row=2, col=2)
        fig.update_xaxes(title_text='Depth', row=2, col=3)
        fig.update_yaxes(title_text='QUAL', row=2, col=3)
        display(fig)

        stats = pd.DataFrame([{
            'total_calls': len(calls),
            'median_depth': round(calls['depth'].median(), 1),
            'mean_depth': round(calls['depth'].mean(), 1),
            'median_qual': round(calls['quality'].median(), 1),
            'pct_af_gt50': round((calls['allele_freq'] > 0.5).mean() * 100, 1)
        }])
        display(stats)
    finally:
        _perrun_busy[0] = False

run_selector.observe(lambda ch: render_perrun(ch['new']), names='value')
render_perrun(run_ids[0])


## Cross-sample comparison
Select two runs to compare their QC metrics side-by-side and see how many variants they share.

In [ ]:
sel_a = widgets.Dropdown(
    options=run_ids, description='Run A:',
    layout=widgets.Layout(width='350px')
)
sel_b = widgets.Dropdown(
    options=run_ids,
    value=run_ids[1] if len(run_ids) > 1 else run_ids[0],
    description='Run B:',
    layout=widgets.Layout(width='350px')
)

_compare_busy = [False]

def render_compare(run_a, run_b):
    if _compare_busy[0]:
        return
    _compare_busy[0] = True
    try:
        clear_output(wait=True)
        display(widgets.HBox([sel_a, sel_b]))

        def get_variants(rid):
            return query("""
                SELECT v.chromosome, v.position, v.ref_allele, v.alt_allele,
                       gc.depth, gc.allele_freq, gc.quality
                FROM genotype_calls gc
                JOIN variants v ON gc.variant_id = v.variant_id
                WHERE gc.run_id = ?
            """, (rid,))

        df_a = get_variants(run_a)
        df_b = get_variants(run_b)

        if df_a.empty or df_b.empty:
            print('One or both runs have no variant data.')
            return

        key_cols = ['chromosome', 'position', 'ref_allele', 'alt_allele']
        set_a = set(df_a[key_cols].itertuples(index=False, name=None))
        set_b = set(df_b[key_cols].itertuples(index=False, name=None))
        shared = len(set_a & set_b)
        only_a = len(set_a - set_b)
        only_b = len(set_b - set_a)

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=('Variant overlap', 'Depth comparison', 'AF comparison'),
            horizontal_spacing=0.12
        )

        labels = [f'{run_a} only', 'Shared', f'{run_b} only']
        values = [only_a, shared, only_b]
        fig.add_trace(go.Bar(
            x=labels, y=values,
            marker_color=['#457b9d', '#2d6a4f', '#e63946'],
            text=[f'{v:,}' for v in values], textposition='auto'
        ), row=1, col=1)

        depth_cap = max(df_a['depth'].quantile(0.99), df_b['depth'].quantile(0.99))
        fig.add_trace(go.Box(
            y=df_a[df_a['depth'] <= depth_cap]['depth'],
            name=run_a, marker_color='#457b9d', boxmean=True
        ), row=1, col=2)
        fig.add_trace(go.Box(
            y=df_b[df_b['depth'] <= depth_cap]['depth'],
            name=run_b, marker_color='#e63946', boxmean=True
        ), row=1, col=2)
        fig.add_trace(go.Histogram(
            x=df_a['allele_freq'], nbinsx=30,
            name=run_a, opacity=0.6, marker_color='#457b9d'
        ), row=1, col=3)
        fig.add_trace(go.Histogram(
            x=df_b['allele_freq'], nbinsx=30,
            name=run_b, opacity=0.6, marker_color='#e63946'
        ), row=1, col=3)

        overlap_pct = shared / len(set_a) * 100 if set_a else 0
        jaccard = shared / len(set_a | set_b) * 100 if (set_a | set_b) else 0
        fig.update_layout(
            height=420, barmode='overlay',
            title=f'{run_a}  vs  {run_b}  |  overlap {overlap_pct:.1f}%  |  Jaccard {jaccard:.1f}%'
        )
        display(fig)

        run_info = query("""
            SELECT r.run_id, s.name AS sample, r.mapping_rate, r.total_raw, r.total_filt
            FROM runs r JOIN samples s ON r.sample_id = s.sample_id
            WHERE r.run_id IN (?, ?)
        """, (run_a, run_b))
        display(run_info.style.format({'mapping_rate': '{:.1f}',
                                       'total_raw': '{:,}', 'total_filt': '{:,}'}))
    finally:
        _compare_busy[0] = False

def on_compare_change(change):
    render_compare(sel_a.value, sel_b.value)

sel_a.observe(on_compare_change, names='value')
sel_b.observe(on_compare_change, names='value')

if len(run_ids) >= 2:
    render_compare(run_ids[0], run_ids[1])


## Variant overlap heatmap
Jaccard similarity between all cell lines — reveals which lines share variants and whether the DB coverage is well-distributed. Only ALT calls are compared.

In [ ]:
import numpy as np

# Load all ALT calls per run
alt_calls = query("""
    SELECT r.run_id, s.cell_line, v.chromosome, v.position, v.ref_allele, v.alt_allele
    FROM genotype_calls gc
    JOIN variants v ON gc.variant_id = v.variant_id
    JOIN runs r ON gc.run_id = r.run_id
    JOIN samples s ON r.sample_id = s.sample_id
    WHERE gc.genotype != '0/0'
""")

# Group by cell_line (or run_id if no cell_line), build variant sets
alt_calls["label"] = alt_calls["cell_line"].fillna(alt_calls["run_id"])
key_cols = ["chromosome", "position", "ref_allele", "alt_allele"]
grouped = alt_calls.groupby("label").apply(
    lambda df: set(map(tuple, df[key_cols].values))
).to_dict()

labels = sorted(grouped.keys())
n = len(labels)
jaccard = np.zeros((n, n))

for i, a in enumerate(labels):
    for j, b in enumerate(labels):
        sa, sb = grouped[a], grouped[b]
        inter = len(sa & sb)
        union = len(sa | sb)
        jaccard[i, j] = inter / union if union else 0

fig = go.Figure(go.Heatmap(
    z=jaccard, x=labels, y=labels,
    colorscale="Blues", zmin=0, zmax=1,
    text=np.round(jaccard, 2),
    texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}<br>Jaccard: %{z:.3f}<extra></extra>"
))
fig.update_layout(
    title="Variant overlap — Jaccard similarity (ALT calls only)",
    height=max(500, 25 * n),
    xaxis_tickangle=45
)
fig.show()


---
## scRNA demultiplexing results
Visualisations for `final_assignments.tsv` produced by the scRNA demux pipeline (cellsnp-lite → Vireo → match_vireo → merge_demux).

In [ ]:
import glob, os

demux_runs = sorted(
    os.path.basename(os.path.dirname(p))
    for p in glob.glob("results/demux/*/final_assignments.tsv")
)

demux_selector = widgets.Dropdown(
    options=demux_runs,
    description="Demux run:",
    layout=widgets.Layout(width="350px")
)
def load_demux(run_id):
    fa = pd.read_csv(f"results/demux/{run_id}/final_assignments.tsv", sep="\t")
    dm_path = f"results/demux/{run_id}/donor_matches.tsv"
    dm = pd.read_csv(dm_path, sep="\t") if os.path.exists(dm_path) else pd.DataFrame()
    return fa, dm

def render_scrna(run_id):
    fa, dm = load_demux(run_id)

    # ── 1. Cell composition bar ──────────────────────────────────────────
    singlets = fa[fa["source"] == "db_match"].copy()
    line_order = (
        singlets.groupby("cell_line").size()
        .sort_values(ascending=False).index.tolist()
    )

    fig1 = go.Figure()
    colors_conf = {"high": "#2d6a4f", "low": "#f4a261"}
    for conf in ["high", "low"]:
        sub = singlets[singlets["match_confidence"] == conf]
        counts = sub["cell_line"].value_counts().reindex(line_order, fill_value=0)
        fig1.add_trace(go.Bar(
            x=counts.index, y=counts.values,
            name=f"confidence: {conf}",
            marker_color=colors_conf[conf]
        ))
    for status, color in [("doublet", "#e63946"), ("unassigned", "#adb5bd")]:
        fig1.add_trace(go.Bar(
            x=[status], y=[(fa["source"] == status).sum()],
            name=status, marker_color=color
        ))

    fig1.update_layout(
        barmode="stack",
        title=f"Cell composition — {run_id}  ({len(fa):,} cells)",
        xaxis_title="Cell line", yaxis_title="Cell count",
        height=420, legend_title="Match confidence"
    )
    display(fig1)

    # ── 2. Doublet pair heatmap ───────────────────────────────────────────
    doublets = fa[fa["source"] == "doublet"].copy()
    dbl_mat = pd.DataFrame(0, index=line_order, columns=line_order)

    for pair_str in doublets["cell_line"].str.replace("^doublet:", "", regex=True):
        parts = [p.strip().replace("vireo:", "") for p in pair_str.split("+")]
        if len(parts) == 2:
            a, b = parts
            if a in dbl_mat.index and b in dbl_mat.columns:
                dbl_mat.loc[a, b] += 1
                dbl_mat.loc[b, a] += 1

    fig2 = go.Figure(go.Heatmap(
        z=dbl_mat.values,
        x=dbl_mat.columns.tolist(),
        y=dbl_mat.index.tolist(),
        colorscale="Reds",
        text=dbl_mat.values,
        texttemplate="%{text}",
        hovertemplate="%{y} + %{x}<br>Doublets: %{z}<extra></extra>"
    ))
    fig2.update_layout(
        title=f"Doublet pair counts — {run_id}  ({len(doublets):,} doublets)",
        height=420, xaxis_tickangle=45
    )
    display(fig2)

    # ── 3. Vireo probability distribution ────────────────────────────────
    prob_df = singlets.copy()
    prob_df["vireo_prob_max"] = pd.to_numeric(prob_df["vireo_prob_max"], errors="coerce")

    fig3 = go.Figure()
    for line in line_order:
        sub = prob_df[prob_df["cell_line"] == line]["vireo_prob_max"].dropna()
        if len(sub) == 0:
            continue
        fig3.add_trace(go.Violin(
            y=sub, name=line, box_visible=True,
            meanline_visible=True, points=False
        ))
    fig3.update_layout(
        title=f"Vireo assignment probability per cell line — {run_id}",
        yaxis_title="prob_max (Vireo confidence)",
        height=420, showlegend=False
    )
    display(fig3)

    # ── 4. Vireo → DB concordance & gap bars ─────────────────────────────
    if not dm.empty and "concordance" in dm.columns:
        dm["concordance"] = pd.to_numeric(dm["concordance"], errors="coerce")
        dm["gap"]         = pd.to_numeric(dm["gap"],         errors="coerce")
        dm["label"]       = dm["cell_line"].fillna(dm["assigned_line"])
        conf_color = {"high": "#2d6a4f", "low": "#f4a261",
                      "no_match": "#e63946", "no_data": "#adb5bd"}
        bar_colors = [conf_color.get(str(c), "#adb5bd") for c in dm["confidence"]]

        fig4 = make_subplots(
            rows=1, cols=2,
            subplot_titles=("Best concordance per donor", "Gap (best − second best)"),
            horizontal_spacing=0.12
        )
        fig4.add_trace(go.Bar(
            x=dm["vireo_donor"], y=dm["concordance"],
            marker_color=bar_colors,
            text=dm["label"], textposition="outside",
            hovertemplate="%{x} → %{text}<br>Concordance: %{y:.3f}<extra></extra>"
        ), row=1, col=1)
        fig4.add_hline(y=0.4, line_dash="dash", line_color="grey",
                       annotation_text="min_concordance", row=1, col=1)

        fig4.add_trace(go.Bar(
            x=dm["vireo_donor"], y=dm["gap"],
            marker_color=bar_colors,
            hovertemplate="%{x}<br>Gap: %{y:.3f}<extra></extra>"
        ), row=1, col=2)
        fig4.add_hline(y=0.1, line_dash="dash", line_color="grey",
                       annotation_text="min_gap", row=1, col=2)

        fig4.update_yaxes(range=[0, 1], row=1, col=1)
        fig4.update_layout(
            title=f"Vireo → DB donor matching — {run_id}",
            height=420, showlegend=False
        )
        display(fig4)
    else:
        print("donor_matches.tsv not found — skipping concordance plot.")

    # ── 5. Scorer vs Vireo agreement ─────────────────────────────────────
    agree_df = singlets[singlets["scorer_assignment"].notna()].copy()
    if not agree_df.empty:
        agree_df["agreement"] = agree_df["agreement"].map({"TRUE": True, "FALSE": False})
        agg = (
            agree_df.groupby("cell_line")["agreement"]
            .value_counts(normalize=True).unstack(fill_value=0)
            .rename(columns={True: "agree", False: "disagree"})
            .reindex(line_order)
        )
        fig5 = go.Figure([
            go.Bar(x=agg.index, y=agg.get("agree",    pd.Series(0, index=agg.index)) * 100,
                   name="Agree",    marker_color="#2d6a4f"),
            go.Bar(x=agg.index, y=agg.get("disagree", pd.Series(0, index=agg.index)) * 100,
                   name="Disagree", marker_color="#e63946"),
        ])
        fig5.update_layout(
            barmode="stack",
            title=f"Vireo vs binomial scorer agreement — {run_id}",
            yaxis_title="% of cells", xaxis_title="Cell line",
            height=380
        )
        display(fig5)
    else:
        print("No scorer data in final_assignments.tsv — run demux.py with the same run_ids first.")


_scrna_busy = [False]

def render_scrna_cell(run_id):
    if _scrna_busy[0]:
        return
    _scrna_busy[0] = True
    try:
        clear_output(wait=True)
        display(demux_selector)
        render_scrna(run_id)
    finally:
        _scrna_busy[0] = False

demux_selector.observe(lambda ch: render_scrna_cell(ch['new']), names='value')

if demux_runs:
    render_scrna_cell(demux_runs[-1])


## STAR alignment QC
Spliced read percentage and short-read fraction across all runs (from STAR log).

In [ ]:
star_qc = query("""
    SELECT r.run_id, s.name AS sample,
           r.mapping_rate
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_date DESC, r.run_id
""")

# Try to pull splice/short metrics from STAR logs directly
import re

def parse_star_log(run_id, sample):
    log_path = f"results/{run_id}/star/{sample}/Log.final.out"
    if not os.path.exists(log_path):
        return None, None
    text = open(log_path).read()
    m_splice = re.search(r"% of reads mapped to multiple loci.*?([\d.]+)%", text)
    m_short  = re.search(r"% of reads unmapped: too short.*?([\d.]+)%", text)
    m_annot  = re.search(r"% of splices: Annotated \(sjdb\).*?([\d.]+)%", text)
    splice = float(m_annot.group(1)) if m_annot else None
    short  = float(m_short.group(1))  if m_short  else None
    return splice, short

star_qc = query("""
    SELECT r.run_id, s.name AS sample, r.mapping_rate
    FROM runs r JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_id
""")

splice_vals, short_vals = [], []
for _, row in star_qc.iterrows():
    sp, sh = parse_star_log(row["run_id"], row["sample"])
    splice_vals.append(sp)
    short_vals.append(sh)

star_qc["annotated_splice_pct"] = splice_vals
star_qc["pct_too_short"] = short_vals

has_star_data = star_qc["annotated_splice_pct"].notna().any()

if has_star_data:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Annotated splice % (higher = better)",
                        "% reads too short (lower = better)"),
        horizontal_spacing=0.12
    )
    fig.add_trace(go.Bar(
        x=star_qc["sample"],
        y=star_qc["annotated_splice_pct"],
        marker_color="#2d6a4f"
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=star_qc["sample"],
        y=star_qc["pct_too_short"],
        marker_color="#e63946"
    ), row=1, col=2)
    fig.update_layout(height=380, showlegend=False, title="STAR alignment QC")
    fig.show()
else:
    print("STAR log files not found or metrics not yet parsed.")
    print("Run the pipeline first, then re-execute this cell.")

display(star_qc)

## Raw SQL explorer
Run any query against the database.

In [ ]:
sql_box = widgets.Textarea(
    value="SELECT s.name, r.run_id, r.mapping_rate, r.total_filt\nFROM runs r JOIN samples s ON r.sample_id = s.sample_id\nORDER BY r.run_date DESC;",
    layout=widgets.Layout(width="100%", height="100px")
)
run_btn = widgets.Button(description="Run query", button_style="primary")
out_sql = widgets.Output()

def on_run_query(btn):
    with out_sql:
        out_sql.clear_output(wait=True)
        try:
            result = query(sql_box.value)
            display(result)
        except Exception as e:
            print(f"Error: {e}")

run_btn.on_click(on_run_query)
display(sql_box, run_btn, out_sql)